In [0]:
# SCD Type 2: Customers Dimension

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, upper, to_date, to_timestamp,
    when, coalesce, lit, current_timestamp, current_date,
    concat_ws, row_number, desc, md5, broadcast, expr
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "ecomstore.ecomlake.bronze_customers"
SILVER_TABLE = "ecomstore.ecomlake.silver_customers_scd"
TEMP_CHANGED_TABLE = "ecomstore.ecomlake.temp_changed_customers"
TEMP_NEW_TABLE = "ecomstore.ecomlake.temp_new_customers"

# DATA QUALITY FIREWALL (Ground Truth Geo Mapping & Defaults)
geo_reference = [
    ("Mumbai", "Maharashtra", "^4000[0-9]{2}$", "400001"),
    ("Delhi", "Delhi", "^1100[0-9]{2}$", "110001"),
    ("Bangalore", "Karnataka", "^560[0-1][0-9]{2}$", "560001"),
    ("Hyderabad", "Telangana", "^5000[0-9]{2}$", "500001"),
    ("Chennai", "Tamil Nadu", "^6000[0-9]{2}$", "600001"),
    ("Pune", "Maharashtra", "^4110[0-9]{2}$", "411001"),
    ("Kolkata", "West Bengal", "^7000[0-9]{2}$", "700001"),
    ("Ahmedabad", "Gujarat", "^3800[0-9]{2}$", "380001"),
    ("Jaipur", "Rajasthan", "^3020[0-9]{2}$", "302001"),
    ("Lucknow", "Uttar Pradesh", "^2260[0-9]{2}$", "226001")
]
geo_ref_df = spark.createDataFrame(geo_reference, ["ref_city", "ref_state", "ref_pincode_regex", "default_pincode"])

bronze_customers = spark.read.table(BRONZE_TABLE)

# 1. Fetch batches chronologically to process incrementally
batches_df = spark.sql(f"SELECT DISTINCT _batch_id FROM {BRONZE_TABLE} ORDER BY _batch_id").collect()
batches = [row['_batch_id'] for row in batches_df]

for current_batch in batches:
    print(f"\n🚀 Processing Batch: {current_batch}")
    
    incoming_batch = bronze_customers.filter(col("_batch_id") == current_batch)

    # Deduplicate WITHIN the current batch only
    w = Window.partitionBy("customer_id").orderBy(desc("updated_at"), desc("_ingestion_timestamp"))
    incoming = (
        incoming_batch
        .withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")
        .withColumn("customer_id", upper(trim(col("customer_id"))))
        .withColumn("first_name", trim(col("first_name")))
        .withColumn("last_name", trim(col("last_name")))
        .withColumn("email", lower(trim(col("email"))))
        .withColumn("city", trim(col("city")))
        .withColumn("state", trim(col("state")))
        .withColumn("pincode", trim(col("pincode")))
        .withColumn("customer_tier",
            when(col("customer_tier").isin("Bronze","Silver","Gold","Platinum"), col("customer_tier"))
            .otherwise(lit("Bronze"))
        )
        .withColumn("gender",
            when(col("gender").isin("M","m","Male","male"), lit("M"))
            .when(col("gender").isin("F","f","Female","female"), lit("F"))
            .otherwise(None)
        )
        .withColumn("registration_date", to_date(col("registration_date")))
        .withColumn("updated_at", to_timestamp(col("updated_at")))
        .select(
            "customer_id", "first_name", "last_name", "email", "phone",
            "city", "state", "pincode", "gender", "customer_tier", "registration_date", "updated_at"
        )
    )

    # 2. Apply Auto-Correction (Imputation)
    cleansed_customers = (
        incoming.alias("stg")
        .join(broadcast(geo_ref_df).alias("ref"), col("stg.city") == col("ref.ref_city"), "left")
        
        # 1: If the state is wrong or missing, overwrite it with the Ground Truth state based on City
        .withColumn("state", coalesce(col("ref.ref_state"), col("stg.state")))
        
        # 2: Check pincode validity using expr() to bypass the PySpark rlike bug
        .withColumn("is_pincode_valid", expr("stg.pincode RLIKE ref.ref_pincode_regex"))
        
        # 3: If pincode is bad, inject the default valid pincode for that city
        .withColumn("pincode",
            when((col("is_pincode_valid") == True) & col("stg.pincode").isNotNull(), col("stg.pincode"))
            .otherwise(coalesce(col("ref.default_pincode"), col("stg.pincode")))
        )
        .drop("ref_city", "ref_state", "ref_pincode_regex", "default_pincode", "is_pincode_valid")
    )

    # Only Route TRUE Structural Failures to Quarantine (e.g., Null Customer IDs)
    bad_customers = cleansed_customers.filter(col("customer_id").isNull())
    
    if bad_customers.count() > 0:
        (
            bad_customers
            .withColumn("rejection_reason", lit("null_customer_id"))
            .write.format("delta").mode("append").option("mergeSchema", "true")
            .saveAsTable(QUARANTINE_TABLE)
        )
        print(f"⚠️  Quarantined {bad_customers.count()} customers for null IDs.")

    # Proceed with Clean & Auto-Corrected Data
    good_customers = cleansed_customers.filter(col("customer_id").isNotNull())

    scd_cols = ["email", "city", "state", "pincode", "customer_tier", "phone"]
    
    incoming_with_hash = good_customers.withColumn(
        "row_hash",
        md5(concat_ws("||", *[coalesce(col(c), lit("__null__")) for c in scd_cols]))
    )

    # 3. Process SCD Type 2 MERGE
    if spark.catalog.tableExists(SILVER_TABLE):
        existing_scd = spark.read.table(SILVER_TABLE)
        current_records = existing_scd.filter(col("is_current") == True)

        raw_changed_records = (
            incoming_with_hash.alias("src")
            .join(current_records.select("customer_id", "row_hash").alias("tgt"), on="customer_id", how="inner")
            .filter(col("src.row_hash") != col("tgt.row_hash"))
            .select("src.*")
        )

        raw_new_records = (
            incoming_with_hash.alias("src")
            .join(current_records.select("customer_id").alias("tgt"), on="customer_id", how="left_anti")
        )

        raw_changed_records.write.format("delta").mode("overwrite").saveAsTable(TEMP_CHANGED_TABLE)
        raw_new_records.write.format("delta").mode("overwrite").saveAsTable(TEMP_NEW_TABLE)

        staged_changed_records = spark.read.table(TEMP_CHANGED_TABLE)
        staged_new_records = spark.read.table(TEMP_NEW_TABLE)

        silver_scd = DeltaTable.forName(spark, SILVER_TABLE)
        (
            silver_scd.alias("target")
            .merge(
                staged_changed_records.alias("source"),
                "target.customer_id = source.customer_id AND target.is_current = true"
            )
            .whenMatchedUpdate(set={
                "is_current": lit(False),
                "end_date":   current_date(),
                "updated_at": current_timestamp()
            })
            .execute()
        )

        def prepare_scd_insert(df):
            return df.select(
                col("customer_id"), col("first_name"), col("last_name"), col("email"), col("phone"),
                col("city"), col("state"), col("pincode"), col("gender"), col("customer_tier"), col("registration_date"),
                col("row_hash"), current_date().alias("start_date"), lit(None).cast("date").alias("end_date"),
                lit(True).alias("is_current"), current_timestamp().alias("updated_at")
            )

        rows_to_insert = staged_changed_records.unionByName(staged_new_records).transform(prepare_scd_insert)
        rows_to_insert.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(SILVER_TABLE)

    else:
        # Initial load
        initial_scd = incoming_with_hash.select(
            col("customer_id"), col("first_name"), col("last_name"), col("email"), col("phone"),
            col("city"), col("state"), col("pincode"), col("gender"), col("customer_tier"), col("registration_date"),
            col("row_hash"), current_date().alias("start_date"), lit(None).cast("date").alias("end_date"),
            lit(True).alias("is_current"), current_timestamp().alias("updated_at")
        )
        initial_scd.write.format("delta").mode("overwrite").option("delta.autoOptimize.optimizeWrite", "true").saveAsTable(SILVER_TABLE)
        print(f"✅ Created initial SCD table")

# Clean up
spark.sql(f"DROP TABLE IF EXISTS {TEMP_CHANGED_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {TEMP_NEW_TABLE}")